In [ ]:
import torch
from typing import Dict, Any
from CCFV.utils.sliding_window_sampling import FeatureExtractor
from pytorch3dunet.unet3d.model import UNet2D
from model_ranking.dataclass import (
    ModelSourceConfig,
)
from model_ranking.models import UnetrWrapper
from model_ranking.feature_ranking import TransferFeatureExtraction

In [7]:
model_config = {
    "source_name": "EPFL",
    "model_name": "E_model_Unetr2",
    "model_type": "UnetrWrapper"
}
model_cfg = ModelSourceConfig.model_validate(model_config)
model_cfg = model_cfg.create_unetr_config(feature_perturbation=None, img_size=256)

In [8]:
model = UnetrWrapper(**model_cfg.model_dump())
print(model)

UnetrWrapper(
  (vit): ViT(
    (patch_embedding): PatchEmbeddingBlock(
      (patch_embeddings): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (blocks): ModuleList(
      (0-11): 12 x TransformerBlock(
        (mlp): MLPBlock(
          (linear1): Linear(in_features=768, out_features=3072, bias=True)
          (linear2): Linear(in_features=3072, out_features=768, bias=True)
          (fn): GELU(approximate='none')
          (drop1): Dropout(p=0.0, inplace=False)
          (drop2): Dropout(p=0.0, inplace=False)
        )
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): SABlock(
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
          (qkv): Linear(in_features=768, out_features=2304, bias=False)
          (to_q): Identity()
          (to_k): Identity()
          (to_v): Identity()
          (input_rearrange): Rearrange('b h (qkv l d) -> qkv b l h d', qkv=3, 

In [ ]:
import numpy as np
from model_ranking.dataclass import (
    EPFLTargetConfig
)
from model_ranking.feature_ranking import (
    TransferFeatureExtraction,
    compute_class_frequencies,
    get_sampling_indices,
    sample_from_image,
)

In [10]:
config: Dict[str, Any] = {
    "target_datasets": [{"name": "EPFL"}],
    "source_models": [
        {"source_name": "EPFL", "model_name": "E_model_Unetr2", "model_type": "UnetrWrapper"}
    ],
    "source_model_base_path": (
        "/g/kreshuk/talks/segmentation_ModelSelection/experiments"
    ),
    "data_base_path": "/scratch/talks/data",
    "feature_cfg": {
        "layers": ["decoder2"],
        "sampling_seed": 42,
        "num_samples": 1000,
        #"output_dir_path": "/g/kreshuk/talks/model_ranking/notebooks/checks",
        "output_dir_path": None,  # Set to None for testing
    },
}

In [ ]:
from model_ranking.dataclass import TransferFeatureExtractionConfig


feature_ranking_cfg = TransferFeatureExtractionConfig.model_validate(config)
feature_ranking = TransferFeatureExtraction(feature_ranking_cfg)

2025-08-06 16:38:36,165 [MainThread] INFO HDF5Dataset - Loading train set from: /scratch/talks/data/EPFL/test.h5...
2025-08-06 16:38:36,165 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
Global mean: 0.5391123294830322, global std: 0.11974480003118515
2025-08-06 16:38:36,182 [MainThread] INFO Dataset - Slice builder config: {'name': 'SliceBuilder', 'patch_shape': (1, 256, 256), 'stride_shape': (1, 256, 256), 'halo_shape': (0, 0, 0)}
2025-08-06 16:38:36,188 [MainThread] INFO HDF5Dataset - Number of patches: 12


In [15]:
target_dataset = feature_ranking.target_datasets["EPFL"]

In [16]:
feature_extractor = FeatureExtractor(model, layers=["decoder2"])
for i, (image, label) in enumerate(iter(target_dataset)):
    print(f"Image {i}: shape={image.shape}, label={label.shape}")
    features = feature_extractor(image)
    break
    

Image 0: shape=torch.Size([1, 1, 256, 256]), label=torch.Size([1, 1, 256, 256])


In [ ]:
features["decoder2"].shape  # Should be (1, 16, 256, 256)

torch.Size([1, 16, 256, 256])

In [1]:
import numpy as np
# EPFL_path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/EPFL_to_EPFL/E_model5_to_EPFL_features.npz"
# Hmito_path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/Hmito_to_Hmito/Hm_model4_to_Hmito_features.npz"
# Rmito_path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/Rmito_to_Rmito/Rm_model4_to_Rmito_features.npz"
VNC_path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/EPFL_to_VNC/E_model5_to_VNC_features.npz"

# EPFL_indices = np.load(EPFL_path)["decoders.2_indices"]
# Hmito_indices = np.load(Hmito_path)["decoders.2_indices"]
# Rmito_indices = np.load(Rmito_path)["decoders.2_indices"]
VNC_indices = np.load(VNC_path)["decoders.2_indices"]

In [2]:
base_indices_path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/feature_indices"
# EPFL_indices_path = f"{base_indices_path}/EPFL_indices.npz"
# Hmito_indices_path = f"{base_indices_path}/Hmito_indices.npz"
# Rmito_indices_path = f"{base_indices_path}/Rmito_indices.npz"
VNC_indices_path = f"{base_indices_path}/VNC_indices.npz"
# np.savez(EPFL_indices_path, **{"decoders.2_indices": EPFL_indices, "decoder2_indices": EPFL_indices, "decoders.3_indices": EPFL_indices})
# np.savez(Hmito_indices_path, **{"decoders.2_indices": Hmito_indices, "decoder2_indices": Hmito_indices, "decoders.3_indices": Hmito_indices})
# np.savez(Rmito_indices_path, **{"decoders.2_indices": Rmito_indices, "decoder2_indices": Rmito_indices, "decoders.3_indices": Rmito_indices})
np.savez(VNC_indices_path, **{"decoders.2_indices": VNC_indices, "decoder2_indices": VNC_indices, "decoders.3_indices": VNC_indices})

In [3]:
path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/feature_indices/EPFL_indices.npz"
indices = np.load(path)["decoders.3_indices"]
print(np.all(np.equal(indices, EPFL_indices)))  # Should be True

True


In [72]:
EPFL_path2 = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/Rmito_to_EPFL/Rm_model_NA2_to_EPFL_features.npz"
Hmito_path2 = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/Rmito_to_Hmito/Rm_model_NA2_to_Hmito_features.npz"
Rmito_path2 = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria/Rmito_to_Rmito/Rm_model_NA2_to_Rmito_features.npz"

EPFL_indices2 = np.load(EPFL_path2)["decoders.2_indices"]
Hmito_indices2 = np.load(Hmito_path)["decoders.2_indices"]
Rmito_indices2 = np.load(Rmito_path2)["decoders.2_indices"]

# Check if the indices match
print(np.all(np.equal(EPFL_indices, EPFL_indices2)))  # Should be True
print(np.all(np.equal(Hmito_indices, Hmito_indices2)))  # Should be True
print(np.all(np.equal(Rmito_indices, Rmito_indices2)))  # Should

True
True
True


In [60]:
from pathlib import Path
base_path = Path("/scratch/talks/sampled_features/semantic_segmentation/mitochondria")
Hmito_target_paths = list(base_path.rglob("**/*_to_Hmito/*_to_Hmito_features.npz"))
Rmito_target_paths = list(base_path.rglob("**/*_to_Rmito/*_to_Rmito_features.npz"))
EPFL_target_paths = list(base_path.rglob("**/*_to_EPFL/*_to_EPFL_features.npz"))
VNC_target_paths = list(base_path.rglob("**/*_to_VNC/*_to_VNC_features.npz"))

In [65]:
non_equal_pairs_Hmito = []
indices_list_Hmito = [np.load(str(path))["decoders.2_indices"] for path in Hmito_target_paths]

for i in range(len(indices_list_Hmito)):
    for j in range(i + 1, len(indices_list_Hmito)):
        if not np.array_equal(indices_list_Hmito[i], indices_list_Hmito[j]):
            non_equal_pairs_Hmito.append((i, j))
print("Hmito:", non_equal_pairs_Hmito)

Hmito: []


In [66]:
non_equal_pairs_Rmito = []
indices_list_Rmito = [np.load(str(path))["decoders.2_indices"] for path in Rmito_target_paths]

for i in range(len(indices_list_Rmito)):
    for j in range(i + 1, len(indices_list_Rmito)):
        if not np.array_equal(indices_list_Rmito[i], indices_list_Rmito[j]):
            non_equal_pairs_Rmito.append((i, j))
print("Rmito:", non_equal_pairs_Rmito)

Rmito: []


In [67]:
print(Rmito_target_paths[6])

/scratch/talks/sampled_features/semantic_segmentation/mitochondria/Rmito_to_Rmito/Rm_model4_to_Rmito_features.npz


In [68]:
non_equal_pairs_EPFL = []
indices_list_EPFL = [np.load(str(path))["decoders.2_indices"] for path in EPFL_target_paths]

for i in range(len(indices_list_EPFL)):
    for j in range(i + 1, len(indices_list_EPFL)):
        if not np.array_equal(indices_list_EPFL[i], indices_list_EPFL[j]):
            non_equal_pairs_EPFL.append((i, j))
print("EPFL:", non_equal_pairs_EPFL)

EPFL: []


In [69]:
print(EPFL_target_paths[1])

/scratch/talks/sampled_features/semantic_segmentation/mitochondria/EPFL_to_EPFL/E_model5_to_EPFL_features.npz


In [70]:
non_equal_pairs_VNC = []
indices_list_VNC = [np.load(str(path))["decoders.2_indices"] for path in VNC_target_paths]

for i in range(len(indices_list_VNC)):
    for j in range(i + 1, len(indices_list_VNC)):
        if not np.array_equal(indices_list_VNC[i], indices_list_VNC[j]):
            non_equal_pairs_VNC.append((i, j))
print("VNC:", non_equal_pairs_VNC)

VNC: []


In [41]:
print(VNC_target_paths[0])

/scratch/talks/sampled_features/semantic_segmentation/mitochondria/EPFL_to_VNC/E_model5_to_VNC_features.npz


In [20]:
EPFL_indices2.shape

(750, 1000)

In [ ]:
EPFL_indices.shape

(750, 1000)

In [57]:
non_matching_ids_EPFL = []
id = 1
for i in range(len(indices_list_EPFL[0])):
    if np.all(np.equal(indices_list_EPFL[id][i], indices_list_EPFL[id+1][i])) == False:
        non_matching_ids_EPFL.append(i)
print(f"Non-matching indices: {non_matching_ids_EPFL}")  # Should be empty if all match
print(len(non_matching_ids_EPFL))  # Should be 0 if all match

Non-matching indices: [736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749]
14


In [58]:
non_matching_ids_Hmito = []
id = 4
for i in range(len(indices_list_Hmito[0])):
    if np.all(np.equal(indices_list_Hmito[id][i], indices_list_Hmito[id+1][i])) == False:
        non_matching_ids_Hmito.append(i)
print(f"Non-matching indices: {non_matching_ids_Hmito}")  # Should be empty if all match
print(len(non_matching_ids_Hmito))  # Should be 0 if all match

Non-matching indices: [3744, 3745, 3746, 3747, 3748, 3749]
6


In [59]:
3750 - (3750//32)*32

6

In [ ]:
 - 5*32

20

In [43]:
print(len(non_matching_ids))

20
